# Tutorial F (Part 1): Building Neural Networks Step by Step

**Understanding neural networks by building them from the ground up**

---

## References and Further Resources

### Key References
- Nielsen, M. (2015). *Neural Networks and Deep Learning*. Determination Press. Chapter 1: Using neural nets to recognize handwritten digits. [Online](http://neuralnetworksanddeeplearning.com/)
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press. Chapter 6: Deep Feedforward Networks.
- Trask, A. (2019). *Grokking Deep Learning*. Manning Publications. Excellent beginner-friendly explanations.

### Further Exploration
- 3Blue1Brown: "Neural Networks" video series - outstanding visual explanations
- Karpathy, A. "The Spelled-Out Intro to Neural Networks and Backpropagation" [YouTube]
- PyTorch tutorials: [pytorch.org/tutorials](https://pytorch.org/tutorials/)

---

## Table of Contents

1. [Introduction: What We're Building and Why](#introduction)
2. [The Simplest Neuron: A Single Perceptron](#single-neuron)
3. [Forward Pass: How Predictions Happen](#forward-pass)
4. [Backward Pass: How Learning Happens](#backward-pass)
5. [The Problem with Functions: Why We Need OOP](#why-oop)
6. [Building Our First Layer Class](#layer-class)
7. [From One Layer to Multiple Layers](#multiple-layers)
8. [Activation Functions: Adding Nonlinearity](#activations)
9. [Putting It Together: A Complete Example](#complete-example)
10. [How This Mirrors Professional Frameworks](#frameworks)
11. [Summary and Next Steps](#summary)

---

## 1. Introduction: What We're Building and Why <a id='introduction'></a>

### The Big Picture

Neural networks are just mathematical functions that:
1. Take inputs (like pixel values of an image)
2. Transform them through layers of calculations
3. Produce outputs (like "this is a cat" or "this is a dog")

The magic is that they **learn** these transformations from examples!

### Our Journey

We'll start with the absolute simplest version and gradually build up:

1. **Single neuron** (1 input → 1 output)
2. **Multiple inputs** (many inputs → 1 output)
3. **Multiple neurons** (many inputs → many outputs)
4. **Multiple layers** (deep networks)

At each step, we'll:
- Show the **function-based** approach first
- See where it gets messy
- Introduce the **OOP approach**
- Explain why it's better

### Why This Matters

When you use PyTorch or TensorFlow, you're using code that looks like:

```python
model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 1)
)
```

By the end of this tutorial, you'll understand *exactly* what's happening inside those classes and why they're designed that way!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List

# Set random seed for reproducibility
np.random.seed(42)

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)

---

## 2. The Simplest Neuron: A Single Perceptron <a id='single-neuron'></a>

Let's start with the absolute simplest neural network: **one input, one weight, one output**.

### The Math

A single neuron computes:
$$y = w \cdot x + b$$

Where:
- $x$ is the input
- $w$ is the weight (how important the input is)
- $b$ is the bias (shifts the output)
- $y$ is the output

This is just a line! Like $y = mx + b$ from algebra.

### Let's Code It (Function Approach)

In [ ]:
def simple_neuron(input_value: float, weight: float, bias: float) -> float:
    """A single neuron: y = w*x + b"""
    return weight * input_value + bias

# Example: Let's say we're predicting temperature from time of day
time_of_day = 14.0  # 2 PM (using 24-hour time)
weight = 1.5        # Temperature increases 1.5°C per hour after midnight
bias = 10.0         # Base temperature at midnight

predicted_temp = simple_neuron(time_of_day, weight, bias)
print(f"At {time_of_day}:00, predicted temperature: {predicted_temp}°C")

# Let's visualize how this neuron behaves
times = np.linspace(0, 24, 100)
temps = [simple_neuron(t, weight, bias) for t in times]

plt.figure(figsize=(10, 6))
plt.plot(times, temps, 'b-', linewidth=2, label=f'y = {weight}*x + {bias}')
plt.scatter([time_of_day], [predicted_temp], color='red', s=100, 
           zorder=5, label='Our prediction')
plt.xlabel('Time of Day (hours)')
plt.ylabel('Temperature (°C)')
plt.title('Single Neuron: Linear Relationship')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nNotice: This neuron draws a straight line!")

### Multiple Inputs: More Realistic

Real problems have multiple inputs. For example, predicting temperature might depend on:
- Time of day
- Day of year
- Cloud cover

Now our neuron computes:
$$y = w_1 x_1 + w_2 x_2 + w_3 x_3 + b$$

Or more compactly:
$$y = \sum_{i=1}^{n} w_i x_i + b = \mathbf{w} \cdot \mathbf{x} + b$$

In [ ]:
def multi_input_neuron(inputs: np.ndarray, weights: np.ndarray, bias: float) -> float:
    """
    A neuron with multiple inputs.
    
    Args:
        inputs: Array of input values [x1, x2, x3, ...]
        weights: Array of weights [w1, w2, w3, ...]
        bias: Bias term
    
    Returns:
        Output value
    """
    return np.dot(weights, inputs) + bias

# Example: Predicting if a student will pass an exam
# Inputs: [hours_studied, previous_grade, attendance_rate]
student_data = np.array([5.0, 75.0, 0.9])  # 5 hours, 75% grade, 90% attendance

# Weights: how important is each factor?
weights = np.array([3.0, 0.5, 10.0])  # Studying and attendance are important!
bias = -20.0

prediction = multi_input_neuron(student_data, weights, bias)
print(f"Student data: {student_data}")
print(f"Weights: {weights}")
print(f"Prediction score: {prediction:.2f}")
print(f"\nBreakdown:")
print(f"  Hours studied contribution: {weights[0]} × {student_data[0]} = {weights[0] * student_data[0]:.2f}")
print(f"  Previous grade contribution: {weights[1]} × {student_data[1]} = {weights[1] * student_data[1]:.2f}")
print(f"  Attendance contribution: {weights[2]} × {student_data[2]} = {weights[2] * student_data[2]:.2f}")
print(f"  Bias: {bias:.2f}")
print(f"  Total: {prediction:.2f}")

### Key Insight: Neurons Compute Weighted Sums

A neuron is just computing a **weighted sum** of its inputs:
- Each input gets multiplied by its weight (importance)
- All contributions are added together
- The bias shifts the result

**Geometric intuition**: In 2D, this is a line. In 3D, it's a plane. In higher dimensions, it's a hyperplane that divides the space.

---

## 3. Forward Pass: How Predictions Happen <a id='forward-pass'></a>

The **forward pass** is the process of computing the output given an input. Let's trace through it step by step.

### Example: Simple AND Gate

Let's build a neuron that learns the AND logic gate:
- Input: (0, 0) → Output: 0
- Input: (0, 1) → Output: 0
- Input: (1, 0) → Output: 0
- Input: (1, 1) → Output: 1

In [ ]:
def forward_pass_example() -> None:
    """Demonstrate a forward pass through a single neuron."""
    
    # Hand-crafted weights for AND gate
    weights = np.array([1.0, 1.0])  # Both inputs matter equally
    bias = -1.5  # Need both inputs to be 1 to overcome this
    
    # All possible inputs for AND gate
    inputs = [
        np.array([0.0, 0.0]),
        np.array([0.0, 1.0]),
        np.array([1.0, 0.0]),
        np.array([1.0, 1.0])
    ]
    
    print("Forward Pass Through AND Gate Neuron")
    print("=" * 50)
    
    for input_vec in inputs:
        # Step 1: Compute weighted sum
        weighted_sum = np.dot(weights, input_vec)
        
        # Step 2: Add bias
        linear_output = weighted_sum + bias
        
        # Step 3: Apply activation (we'll use step function for now)
        # If output > 0, predict 1, else predict 0
        output = 1.0 if linear_output > 0 else 0.0
        
        print(f"\nInput: {input_vec}")
        print(f"  Step 1 - Weighted sum: {weights[0]}*{input_vec[0]} + {weights[1]}*{input_vec[1]} = {weighted_sum:.2f}")
        print(f"  Step 2 - Add bias: {weighted_sum:.2f} + {bias:.2f} = {linear_output:.2f}")
        print(f"  Step 3 - Activation: {linear_output:.2f} > 0? → {int(output)}")
        print(f"  Expected: {int(input_vec[0] and input_vec[1])} ✓" if output == (input_vec[0] and input_vec[1]) else "  Expected: {int(input_vec[0] and input_vec[1])} ✗")

forward_pass_example()

### Visualizing the Forward Pass

Let's see what's happening geometrically:

In [ ]:
def visualize_decision_boundary() -> None:
    """Visualize how a neuron divides the input space."""
    
    # Create a grid of points
    x1_range = np.linspace(-0.5, 1.5, 100)
    x2_range = np.linspace(-0.5, 1.5, 100)
    X1, X2 = np.meshgrid(x1_range, x2_range)
    
    # AND gate weights
    weights = np.array([1.0, 1.0])
    bias = -1.5
    
    # Compute output for each point
    Z = weights[0] * X1 + weights[1] * X2 + bias
    
    # Plot
    plt.figure(figsize=(10, 8))
    
    # Contour plot showing the decision boundary
    plt.contourf(X1, X2, Z, levels=[-10, 0, 10], colors=['lightblue', 'lightcoral'], alpha=0.6)
    plt.contour(X1, X2, Z, levels=[0], colors=['black'], linewidths=3)
    
    # Plot the data points
    points = [
        ([0, 0], 0, 'Class 0'),
        ([0, 1], 0, 'Class 0'),
        ([1, 0], 0, 'Class 0'),
        ([1, 1], 1, 'Class 1')
    ]
    
    for idx, (point, label, name) in enumerate(points):
        color = 'red' if label == 1 else 'blue'
        marker = 'o' if label == 1 else 's'
        plt.scatter(point[0], point[1], c=color, marker=marker, s=200, 
                   edgecolors='black', linewidths=2, zorder=5,
                   label=name if idx == 0 or idx == 3 else '')
    
    plt.xlabel('Input 1', fontsize=12)
    plt.ylabel('Input 2', fontsize=12)
    plt.title('How a Neuron Divides Input Space\n(Decision Boundary in Black)', fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    
    # Add annotations
    plt.text(0.2, 0.2, 'Predicts 0\n(Blue region)', fontsize=11, ha='center')
    plt.text(1.2, 1.2, 'Predicts 1\n(Red region)', fontsize=11, ha='center')
    
    plt.xlim(-0.5, 1.5)
    plt.ylim(-0.5, 1.5)
    plt.show()

visualize_decision_boundary()

print("\nKey insight: The neuron draws a line (decision boundary) that separates the two classes!")

### Batch Processing: Multiple Examples at Once

In practice, we process many examples simultaneously (a "batch"). This is faster and more efficient.

Instead of:
- Input: single vector `[x1, x2]`
- Output: single number

We have:
- Input: matrix where each **row** is an example
- Output: vector with one value per example

In [ ]:
def batch_forward_pass(inputs_batch: np.ndarray, 
                       weights: np.ndarray, 
                       bias: float) -> np.ndarray:
    """
    Forward pass for a batch of inputs.
    
    Args:
        inputs_batch: Shape (batch_size, num_inputs)
        weights: Shape (num_inputs,)
        bias: Scalar
    
    Returns:
        outputs: Shape (batch_size,)
    """
    # Matrix multiplication: each row of inputs_batch gets multiplied by weights
    return inputs_batch @ weights + bias

# Example: Process all AND gate inputs at once
inputs_batch = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0]
])

weights = np.array([1.0, 1.0])
bias = -1.5

outputs = batch_forward_pass(inputs_batch, weights, bias)

print("Batch Forward Pass")
print("=" * 50)
print(f"Input batch shape: {inputs_batch.shape}")
print(f"Weights shape: {weights.shape}")
print(f"Output shape: {outputs.shape}")
print(f"\nInputs:\n{inputs_batch}")
print(f"\nOutputs: {outputs}")
print(f"\nPredictions (after threshold): {(outputs > 0).astype(int)}")
print(f"Expected: [0, 0, 0, 1]")

print("\n✓ All predictions correct!")

### Summary of Forward Pass

The forward pass is:
1. **Multiply**: Each input by its weight
2. **Sum**: Add all weighted inputs
3. **Shift**: Add the bias
4. **Activate**: Apply a nonlinear function (optional, we'll add this soon)

In code: `output = activation(inputs @ weights + bias)`

---

## 4. Backward Pass: How Learning Happens <a id='backward-pass'></a>

The forward pass makes predictions. But how do we **learn** good weights?

### The Learning Problem

Given:
- Training data: inputs and correct outputs
- A random initialization of weights

Goal:
- Adjust weights to make better predictions

How:
- **Measure error**: How wrong are we?
- **Compute gradients**: Which direction should we adjust each weight?
- **Update weights**: Move in the direction that reduces error

### A Simple Example: Learning by Trial and Error

In [ ]:
def learn_simple_relationship() -> None:
    """
    Learn a simple relationship: y = 2*x using gradient descent.
    
    This shows the backward pass in its simplest form.
    """
    
    # True relationship: y = 2*x (we're trying to learn weight = 2)
    true_weight = 2.0
    
    # Generate training data
    num_samples = 20
    x_train = np.random.uniform(0, 10, num_samples)
    y_train = true_weight * x_train + np.random.normal(0, 0.5, num_samples)
    
    # Start with a random guess
    weight = 0.5
    learning_rate = 0.01
    
    # Track learning progress
    weight_history = [weight]
    error_history = []
    
    print("Learning Process")
    print("=" * 60)
    print(f"True weight: {true_weight}")
    print(f"Starting weight: {weight}\n")
    
    # Learning loop
    num_iterations = 100
    for iteration in range(num_iterations):
        # FORWARD PASS: Make predictions
        predictions = weight * x_train
        
        # COMPUTE ERROR: How wrong are we?
        errors = predictions - y_train
        total_error = np.mean(errors ** 2)  # Mean squared error
        error_history.append(total_error)
        
        # BACKWARD PASS: Compute gradient
        # Gradient of MSE with respect to weight:
        # d(MSE)/d(weight) = 2 * mean(error * x)
        gradient = 2 * np.mean(errors * x_train)
        
        # UPDATE WEIGHT: Move in direction that reduces error
        weight = weight - learning_rate * gradient
        weight_history.append(weight)
        
        # Print progress
        if iteration % 20 == 0:
            print(f"Iteration {iteration:3d}: weight = {weight:.4f}, error = {total_error:.4f}")
    
    print(f"\nFinal weight: {weight:.4f}")
    print(f"True weight: {true_weight:.4f}")
    print(f"Difference: {abs(weight - true_weight):.4f}")
    
    # Visualize learning
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Weight convergence
    ax1.plot(weight_history, 'b-', linewidth=2)
    ax1.axhline(y=true_weight, color='r', linestyle='--', linewidth=2, label='True weight')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Weight Value')
    ax1.set_title('Weight Learning Over Time')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Error reduction
    ax2.plot(error_history, 'g-', linewidth=2)
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Mean Squared Error')
    ax2.set_title('Error Decreasing Over Time')
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

learn_simple_relationship()

### Understanding Gradients

The **gradient** tells us:
- **Direction**: Should we increase or decrease the weight?
- **Magnitude**: How much should we change it?

Think of it like this:
- You're in a foggy mountain trying to get to the bottom
- You can't see far, but you can feel the slope
- The gradient is the direction of steepest descent
- You take small steps downhill (that's the learning rate)

### The Chain Rule: Gradients in Deep Networks

For a simple neuron, computing gradients is easy. But what about multiple layers?

That's where **backpropagation** comes in - it uses the chain rule from calculus to compute gradients efficiently.

Let's see a two-step example:

In [ ]:
def demonstrate_chain_rule() -> None:
    """
    Show how the chain rule works in a simple 2-step computation.
    
    Computation: x → [multiply by w1] → h → [multiply by w2] → y
    Loss: L = (y - target)^2
    
    We want: dL/dw1 and dL/dw2
    """
    
    print("Chain Rule Example")
    print("=" * 60)
    print("Computation graph: x → [×w1] → h → [×w2] → y")
    print("Loss: L = (y - target)^2\n")
    
    # Values
    x = 2.0
    target = 12.0
    w1 = 1.5
    w2 = 3.0
    
    print("Forward pass:")
    print("-" * 60)
    
    # Step 1: Multiply by w1
    h = w1 * x
    print(f"h = w1 * x = {w1} * {x} = {h}")
    
    # Step 2: Multiply by w2
    y = w2 * h
    print(f"y = w2 * h = {w2} * {h} = {y}")
    
    # Compute loss
    loss = (y - target) ** 2
    print(f"L = (y - target)^2 = ({y} - {target})^2 = {loss}")
    
    print("\nBackward pass (computing gradients):")
    print("-" * 60)
    
    # Gradient of loss with respect to y
    dL_dy = 2 * (y - target)
    print(f"dL/dy = 2*(y - target) = 2*({y} - {target}) = {dL_dy}")
    
    # Gradient with respect to w2 (using chain rule)
    # dL/dw2 = dL/dy * dy/dw2
    dy_dw2 = h  # Since y = w2 * h, dy/dw2 = h
    dL_dw2 = dL_dy * dy_dw2
    print(f"\ndL/dw2 = dL/dy * dy/dw2")
    print(f"       = {dL_dy} * {dy_dw2}")
    print(f"       = {dL_dw2}")
    
    # Gradient with respect to h (needed for w1)
    # dL/dh = dL/dy * dy/dh
    dy_dh = w2  # Since y = w2 * h, dy/dh = w2
    dL_dh = dL_dy * dy_dh
    print(f"\ndL/dh = dL/dy * dy/dh")
    print(f"      = {dL_dy} * {dy_dh}")
    print(f"      = {dL_dh}")
    
    # Gradient with respect to w1 (using chain rule through h)
    # dL/dw1 = dL/dh * dh/dw1
    dh_dw1 = x  # Since h = w1 * x, dh/dw1 = x
    dL_dw1 = dL_dh * dh_dw1
    print(f"\ndL/dw1 = dL/dh * dh/dw1")
    print(f"       = {dL_dh} * {dh_dw1}")
    print(f"       = {dL_dw1}")
    
    print("\n" + "=" * 60)
    print("Key insight: Gradients flow backward through the computation!")
    print("Each layer receives gradient from the next layer and passes it back.")
    
    # Update weights
    learning_rate = 0.01
    w1_new = w1 - learning_rate * dL_dw1
    w2_new = w2 - learning_rate * dL_dw2
    
    print(f"\nAfter update (learning_rate = {learning_rate}):")
    print(f"w1: {w1:.4f} → {w1_new:.4f}")
    print(f"w2: {w2:.4f} → {w2_new:.4f}")
    
    # Verify the update reduces loss
    y_new = w2_new * (w1_new * x)
    loss_new = (y_new - target) ** 2
    print(f"\nLoss: {loss:.4f} → {loss_new:.4f} ✓ (decreased!)")

demonstrate_chain_rule()

### Summary of Backward Pass

The backward pass:
1. **Compute loss**: Measure how wrong predictions are
2. **Compute gradient of loss w.r.t. output**: Start of the chain
3. **Propagate gradients backward**: Use chain rule layer by layer
4. **Update weights**: Move in the direction that reduces loss

This is **backpropagation** - the fundamental algorithm for training neural networks!

---

## 5. The Problem with Functions: Why We Need OOP <a id='why-oop'></a>

So far, we've used functions. Let's see why this becomes problematic as networks get larger.

### Attempt: Building a 3-Layer Network with Functions

In [ ]:
def messy_functional_approach() -> None:
    """
    Show how messy it gets to build a network with just functions.
    """
    
    # Network architecture: 2 → 4 → 4 → 1
    
    # Initialize all weights and biases
    w1 = np.random.randn(2, 4) * 0.01
    b1 = np.zeros(4)
    w2 = np.random.randn(4, 4) * 0.01
    b2 = np.zeros(4)
    w3 = np.random.randn(4, 1) * 0.01
    b3 = np.zeros(1)
    
    # Generate some data
    X = np.random.randn(10, 2)
    y = np.random.randn(10, 1)
    
    # Forward pass
    h1 = X @ w1 + b1
    a1 = np.maximum(0, h1)  # ReLU
    
    h2 = a1 @ w2 + b2
    a2 = np.maximum(0, h2)  # ReLU
    
    h3 = a2 @ w3 + b3
    predictions = h3  # No activation on output
    
    # Compute loss
    loss = np.mean((predictions - y) ** 2)
    
    # Backward pass - this is where it gets REALLY messy
    dL_dpred = 2 * (predictions - y) / len(y)
    
    # Layer 3 gradients
    dL_dw3 = a2.T @ dL_dpred
    dL_db3 = np.sum(dL_dpred, axis=0)
    dL_da2 = dL_dpred @ w3.T
    
    # Layer 2 gradients (through ReLU)
    dL_dh2 = dL_da2 * (h2 > 0)
    dL_dw2 = a1.T @ dL_dh2
    dL_db2 = np.sum(dL_dh2, axis=0)
    dL_da1 = dL_dh2 @ w2.T
    
    # Layer 1 gradients (through ReLU)
    dL_dh1 = dL_da1 * (h1 > 0)
    dL_dw1 = X.T @ dL_dh1
    dL_db1 = np.sum(dL_dh1, axis=0)
    
    # Update weights
    learning_rate = 0.01
    w1 -= learning_rate * dL_dw1
    b1 -= learning_rate * dL_db1
    w2 -= learning_rate * dL_dw2
    b2 -= learning_rate * dL_db2
    w3 -= learning_rate * dL_dw3
    b3 -= learning_rate * dL_db3
    
    print("Problems with the Functional Approach:")
    print("=" * 60)
    print("1. Variables everywhere: w1, b1, w2, b2, w3, b3, h1, a1, h2, a2...")
    print("2. Repetitive code: Similar operations for each layer")
    print("3. Easy to make mistakes: Miss a gradient or use wrong variable")
    print("4. Hard to modify: Want to add a layer? Rewrite everything!")
    print("5. Not reusable: Can't easily use this network elsewhere")
    print("6. Hard to test: How do you test individual pieces?")
    print("\nWith 10 layers, this becomes unmaintainable!")

messy_functional_approach()

### What We Need

We need a way to:
1. **Encapsulate** each layer's weights and logic
2. **Reuse** layer implementations
3. **Compose** layers easily
4. **Hide** complexity (we don't need to see all the gradient math)
5. **Test** individual components

This is exactly what **Object-Oriented Programming** gives us!

---

## 6. Building Our First Layer Class <a id='layer-class'></a>

Let's build a Layer class that encapsulates everything a layer needs.

In [ ]:
class SimpleLayer:
    """
    A simple dense layer that:
    - Stores its own weights and biases
    - Knows how to compute forward pass
    - Knows how to compute backward pass
    """
    
    def __init__(self, input_size: int, output_size: int) -> None:
        """
        Initialize the layer.
        
        Args:
            input_size: Number of inputs to this layer
            output_size: Number of outputs (neurons) in this layer
        """
        # Initialize weights with small random values
        self.weights = np.random.randn(input_size, output_size) * 0.01
        self.biases = np.zeros(output_size)
        
        # We'll need to remember inputs for the backward pass
        self.inputs = None
        
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        """
        Forward pass: compute outputs given inputs.
        
        Args:
            inputs: Input data, shape (batch_size, input_size)
            
        Returns:
            outputs: Output data, shape (batch_size, output_size)
        """
        # Save inputs for backward pass
        self.inputs = inputs
        
        # Compute linear transformation
        outputs = inputs @ self.weights + self.biases
        
        return outputs
    
    def backward(self, output_gradient: np.ndarray, learning_rate: float) -> np.ndarray:
        """
        Backward pass: compute gradients and update weights.
        
        Args:
            output_gradient: Gradient from the next layer, shape (batch_size, output_size)
            learning_rate: How much to adjust weights
            
        Returns:
            input_gradient: Gradient to pass to previous layer, shape (batch_size, input_size)
        """
        # Compute gradients
        batch_size = self.inputs.shape[0]
        
        # Gradient with respect to weights
        weight_gradient = self.inputs.T @ output_gradient / batch_size
        
        # Gradient with respect to biases
        bias_gradient = np.mean(output_gradient, axis=0)
        
        # Gradient with respect to inputs (for previous layer)
        input_gradient = output_gradient @ self.weights.T
        
        # Update weights and biases
        self.weights -= learning_rate * weight_gradient
        self.biases -= learning_rate * bias_gradient
        
        return input_gradient
    
    def __repr__(self) -> str:
        return f"SimpleLayer({self.weights.shape[0]} → {self.weights.shape[1]})"

### Testing Our Layer Class

In [ ]:
# Create a layer: 3 inputs → 2 outputs
layer = SimpleLayer(input_size=3, output_size=2)

print(f"Created: {layer}")
print(f"\nWeights shape: {layer.weights.shape}")
print(f"Biases shape: {layer.biases.shape}")
print(f"\nInitial weights:\n{layer.weights}")
print(f"\nInitial biases: {layer.biases}")

# Forward pass
sample_input = np.array([[1.0, 2.0, 3.0]])  # One example
output = layer.forward(sample_input)

print(f"\nInput: {sample_input}")
print(f"Output: {output}")

# Backward pass (simulate some gradient from next layer)
simulated_gradient = np.array([[0.5, -0.3]])
input_gradient = layer.backward(simulated_gradient, learning_rate=0.1)

print(f"\nGradient from next layer: {simulated_gradient}")
print(f"Gradient for previous layer: {input_gradient}")
print(f"\nWeights after update:\n{layer.weights}")
print("\n✓ The layer handles its own state and updates!")

### Why This Is Better

**With OOP:**
```python
layer = SimpleLayer(3, 2)
output = layer.forward(input)
gradient = layer.backward(output_gradient, lr)
```

**Without OOP:**
```python
w = np.random.randn(3, 2)
b = np.zeros(2)
output = input @ w + b
# ... manually compute all gradients ...
w -= lr * weight_grad
b -= lr * bias_grad
```

Benefits:
1. **Cleaner code**: Layer manages itself
2. **Reusable**: Create as many layers as needed
3. **Testable**: Can test one layer in isolation
4. **Maintainable**: Change implementation without affecting other code

---

## 7. From One Layer to Multiple Layers <a id='multiple-layers'></a>

Now let's stack layers to create a network!

In [ ]:
def build_multilayer_network() -> None:
    """Build and test a multi-layer network using our SimpleLayer class."""
    
    # Create a 3-layer network: 2 → 4 → 4 → 1
    layer1 = SimpleLayer(2, 4)
    layer2 = SimpleLayer(4, 4)
    layer3 = SimpleLayer(4, 1)
    
    print("Network Architecture:")
    print("=" * 60)
    print(f"Input → {layer1} → {layer2} → {layer3} → Output")
    print("        2 → 4        4 → 4        4 → 1")
    
    # Generate sample data
    X = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
    y = np.array([[1.0], [2.0], [3.0]])
    
    print(f"\nTraining data shape: {X.shape}")
    print(f"Target data shape: {y.shape}")
    
    # Training loop
    learning_rate = 0.01
    num_epochs = 100
    losses = []
    
    for epoch in range(num_epochs):
        # FORWARD PASS through all layers
        h1 = layer1.forward(X)
        h2 = layer2.forward(h1)
        predictions = layer3.forward(h2)
        
        # Compute loss
        loss = np.mean((predictions - y) ** 2)
        losses.append(loss)
        
        # BACKWARD PASS through all layers (in reverse)
        loss_gradient = 2 * (predictions - y) / len(y)
        grad3 = layer3.backward(loss_gradient, learning_rate)
        grad2 = layer2.backward(grad3, learning_rate)
        grad1 = layer1.backward(grad2, learning_rate)
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch:3d}, Loss: {loss:.6f}")
    
    # Final predictions
    h1 = layer1.forward(X)
    h2 = layer2.forward(h1)
    final_predictions = layer3.forward(h2)
    
    print("\nFinal Results:")
    print("=" * 60)
    print("Inputs    | Targets | Predictions")
    print("-" * 60)
    for i in range(len(X)):
        print(f"{X[i]}  |  {y[i][0]:.2f}    |  {final_predictions[i][0]:.4f}")
    
    # Plot learning curve
    plt.figure(figsize=(10, 6))
    plt.plot(losses, 'b-', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Loss (MSE)')
    plt.title('Training Progress: Multi-Layer Network')
    plt.yscale('log')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print("\n✓ The network learned! Loss decreased over time.")

build_multilayer_network()

### Code Comparison: OOP vs Functions

Notice how much cleaner the OOP version is:

**Forward pass with OOP:**
```python
h1 = layer1.forward(X)
h2 = layer2.forward(h1)
predictions = layer3.forward(h2)
```

**Forward pass without OOP:**
```python
h1 = X @ w1 + b1
h2 = h1 @ w2 + b2
predictions = h2 @ w3 + b3
```

**Backward pass with OOP:**
```python
grad3 = layer3.backward(loss_gradient, lr)
grad2 = layer2.backward(grad3, lr)
grad1 = layer1.backward(grad2, lr)
```

**Backward pass without OOP:**
```python
# 20+ lines of manual gradient computation
# Easy to make mistakes!
```

---

## 8. Activation Functions: Adding Nonlinearity <a id='activations'></a>

Our network can only learn linear relationships. We need **activation functions** for nonlinearity!

### Why Nonlinearity Matters

In [ ]:
def demonstrate_need_for_nonlinearity() -> None:
    """Show why we need nonlinear activations."""
    
    print("Problem: XOR Gate")
    print("=" * 60)
    print("Inputs  | Output")
    print("--------+-------")
    print(" 0, 0   |   0   ")
    print(" 0, 1   |   1   ")
    print(" 1, 0   |   1   ")
    print(" 1, 1   |   0   ")
    print("\nThis is NOT linearly separable!")
    print("A single linear neuron cannot solve it.")
    
    # Visualize
    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 2, 1)
    plt.scatter([0, 1], [0, 1], c='blue', s=200, marker='s', label='Output = 0')
    plt.scatter([0, 1], [1, 0], c='red', s=200, marker='o', label='Output = 1')
    plt.xlabel('Input 1')
    plt.ylabel('Input 2')
    plt.title('XOR Problem: No Single Line Can Separate These!')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xlim(-0.2, 1.2)
    plt.ylim(-0.2, 1.2)
    
    # Show that we need curved decision boundary
    plt.subplot(1, 2, 2)
    plt.scatter([0, 1], [0, 1], c='blue', s=200, marker='s', label='Output = 0')
    plt.scatter([0, 1], [1, 0], c='red', s=200, marker='o', label='Output = 1')
    
    # Draw a curved boundary (illustration)
    x = np.linspace(-0.2, 1.2, 100)
    y = -x**2 + x + 0.5
    plt.plot(x, y, 'k-', linewidth=2, label='Curved boundary needed')
    
    plt.xlabel('Input 1')
    plt.ylabel('Input 2')
    plt.title('Solution: Nonlinear Boundary')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xlim(-0.2, 1.2)
    plt.ylim(-0.2, 1.2)
    
    plt.tight_layout()
    plt.show()
    
    print("\nSolution: Add nonlinear activation functions!")
    print("With nonlinearity, even a small network can solve XOR.")

demonstrate_need_for_nonlinearity()

### Creating Activation Classes

Let's create classes for common activation functions:

In [ ]:
class ReLU:
    """Rectified Linear Unit: f(x) = max(0, x)"""
    
    def __init__(self) -> None:
        self.inputs = None
    
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        """Apply ReLU activation."""
        self.inputs = inputs
        return np.maximum(0, inputs)
    
    def backward(self, output_gradient: np.ndarray, learning_rate: float) -> np.ndarray:
        """Compute gradient through ReLU."""
        # Gradient is 1 where input > 0, else 0
        return output_gradient * (self.inputs > 0)
    
    def __repr__(self) -> str:
        return "ReLU()"


class Sigmoid:
    """Sigmoid: f(x) = 1 / (1 + e^(-x))"""
    
    def __init__(self) -> None:
        self.outputs = None
    
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        """Apply sigmoid activation."""
        self.outputs = 1 / (1 + np.exp(-np.clip(inputs, -500, 500)))
        return self.outputs
    
    def backward(self, output_gradient: np.ndarray, learning_rate: float) -> np.ndarray:
        """Compute gradient through sigmoid."""
        # Derivative: sigmoid(x) * (1 - sigmoid(x))
        return output_gradient * self.outputs * (1 - self.outputs)
    
    def __repr__(self) -> str:
        return "Sigmoid()"


# Test the activations
x = np.array([[-2, -1, 0, 1, 2]])

relu = ReLU()
sigmoid = Sigmoid()

print("Activation Functions in Action")
print("=" * 60)
print(f"Input: {x[0]}")
print(f"\nReLU output: {relu.forward(x)[0]}")
print(f"Sigmoid output: {sigmoid.forward(x)[0]}")

# Visualize
x_plot = np.linspace(-5, 5, 100).reshape(-1, 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(x_plot, relu.forward(x_plot), 'b-', linewidth=2, label='ReLU')
plt.axhline(0, color='k', linestyle='--', alpha=0.3)
plt.axvline(0, color='k', linestyle='--', alpha=0.3)
plt.title('ReLU Activation')
plt.xlabel('Input')
plt.ylabel('Output')
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(x_plot, sigmoid.forward(x_plot), 'r-', linewidth=2, label='Sigmoid')
plt.axhline(0.5, color='k', linestyle='--', alpha=0.3)
plt.axvline(0, color='k', linestyle='--', alpha=0.3)
plt.title('Sigmoid Activation')
plt.xlabel('Input')
plt.ylabel('Output')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

print("\nNotice:")
print("- ReLU: Simple, fast, most popular in deep learning")
print("- Sigmoid: Smooth, outputs in (0,1), good for probabilities")

---

## 9. Putting It Together: A Complete Example <a id='complete-example'></a>

Let's solve XOR using our layer and activation classes!

In [ ]:
def solve_xor_with_oop() -> None:
    """Solve XOR problem using our OOP neural network components."""
    
    # XOR data
    X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
    y = np.array([[0], [1], [1], [0]])
    
    print("Solving XOR with Neural Network")
    print("=" * 60)
    print("Network: 2 → [Dense(4)] → [ReLU] → [Dense(1)] → [Sigmoid]")
    print()
    
    # Build network
    layer1 = SimpleLayer(2, 4)
    activation1 = ReLU()
    layer2 = SimpleLayer(4, 1)
    activation2 = Sigmoid()
    
    # Training
    learning_rate = 0.5
    num_epochs = 5000
    losses = []
    
    for epoch in range(num_epochs):
        # Forward pass
        h1 = layer1.forward(X)
        a1 = activation1.forward(h1)
        h2 = layer2.forward(a1)
        predictions = activation2.forward(h2)
        
        # Compute loss
        loss = np.mean((predictions - y) ** 2)
        losses.append(loss)
        
        # Backward pass
        loss_grad = 2 * (predictions - y) / len(y)
        grad_act2 = activation2.backward(loss_grad, learning_rate)
        grad_layer2 = layer2.backward(grad_act2, learning_rate)
        grad_act1 = activation1.backward(grad_layer2, learning_rate)
        grad_layer1 = layer1.backward(grad_act1, learning_rate)
        
        if epoch % 1000 == 0:
            print(f"Epoch {epoch:4d}, Loss: {loss:.6f}")
    
    # Final predictions
    h1 = layer1.forward(X)
    a1 = activation1.forward(h1)
    h2 = layer2.forward(a1)
    final_predictions = activation2.forward(h2)
    
    print("\nFinal Results:")
    print("=" * 60)
    print("Input    | Target | Prediction | Correct?")
    print("-" * 60)
    for i in range(len(X)):
        pred_class = 1 if final_predictions[i][0] > 0.5 else 0
        correct = "✓" if pred_class == y[i][0] else "✗"
        print(f"{X[i]}  |   {y[i][0]}    |   {final_predictions[i][0]:.4f}   |   {correct}")
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Learning curve
    ax1.plot(losses, 'b-', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Progress')
    ax1.set_yscale('log')
    ax1.grid(True, alpha=0.3)
    
    # Decision regions
    x1_range = np.linspace(-0.5, 1.5, 100)
    x2_range = np.linspace(-0.5, 1.5, 100)
    X1, X2 = np.meshgrid(x1_range, x2_range)
    grid_points = np.c_[X1.ravel(), X2.ravel()]
    
    # Predict for all grid points
    h1 = layer1.forward(grid_points)
    a1 = activation1.forward(h1)
    h2 = layer2.forward(a1)
    Z = activation2.forward(h2)
    Z = Z.reshape(X1.shape)
    
    ax2.contourf(X1, X2, Z, levels=20, cmap='RdBu', alpha=0.6)
    ax2.contour(X1, X2, Z, levels=[0.5], colors=['black'], linewidths=3)
    ax2.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap='RdBu', 
               s=200, edgecolors='black', linewidths=2)
    ax2.set_xlabel('Input 1')
    ax2.set_ylabel('Input 2')
    ax2.set_title('Learned Decision Boundary')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Network learned XOR! The nonlinear boundary separates the classes.")

solve_xor_with_oop()

### What We Just Did

1. **Created layers** that manage their own weights
2. **Added activations** for nonlinearity
3. **Composed them** into a network
4. **Trained successfully** on a non-linear problem

And the code is **clean and readable**!

---

## 10. How This Mirrors Professional Frameworks <a id='frameworks'></a>

### Our Code vs PyTorch

**Our approach:**
```python
layer1 = SimpleLayer(2, 4)
activation1 = ReLU()
layer2 = SimpleLayer(4, 1)
activation2 = Sigmoid()

# Forward
h1 = layer1.forward(X)
a1 = activation1.forward(h1)
h2 = layer2.forward(a1)
output = activation2.forward(h2)
```

**PyTorch:**
```python
model = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1),
    nn.Sigmoid()
)

# Forward
output = model(X)
```

### Key Parallels

| Our Implementation | PyTorch | Purpose |
|-------------------|---------|----------|
| `SimpleLayer` | `nn.Linear` | Dense layer |
| `ReLU` | `nn.ReLU` | Activation |
| `forward()` | `forward()` | Forward pass |
| `backward()` | Auto-computed | Gradients |

### What PyTorch Adds

1. **Automatic differentiation**: No manual backward pass!
2. **GPU support**: Train on graphics cards (100x faster)
3. **More layers**: Convolutions, recurrent, attention, etc.
4. **Optimizers**: Better update rules than simple gradient descent
5. **Data loaders**: Efficient batch processing
6. **Pretrained models**: Start with models trained on huge datasets

But the **core concepts** are exactly what we built!

In [ ]:
print("Comparison: Manual vs Framework")
print("=" * 60)
print("\nWhat we built:")
print("  ✓ Layers that manage weights")
print("  ✓ Forward pass computation")
print("  ✓ Backward pass (manual gradients)")
print("  ✓ Activation functions")
print("  ✓ Training loop")
print("\nWhat PyTorch/TensorFlow add:")
print("  ✓ Automatic differentiation (autograd)")
print("  ✓ GPU acceleration")
print("  ✓ Advanced optimizers (Adam, RMSprop, etc.)")
print("  ✓ Many more layer types")
print("  ✓ Efficient data loading")
print("  ✓ Model saving/loading")
print("  ✓ Debugging tools")
print("\nBut the fundamental design is the same!")
print("\nUnderstanding what we built helps you:")
print("  • Debug training problems")
print("  • Design custom layers")
print("  • Understand framework documentation")
print("  • Make informed architecture choices")

---

## 11. Summary and Next Steps <a id='summary'></a>

### What We've Learned

**Core Concepts:**
1. **Neurons**: Compute weighted sums of inputs
2. **Forward pass**: How predictions are made
3. **Backward pass**: How learning happens (backpropagation)
4. **Activations**: Add nonlinearity for complex patterns
5. **Layers**: Building blocks of networks

**Why OOP Matters:**
1. **Encapsulation**: Each layer manages its own state
2. **Reusability**: Use the same layer class many times
3. **Composition**: Stack layers to build deep networks
4. **Maintainability**: Easy to modify and debug
5. **Mirrors frameworks**: Same design as PyTorch/TensorFlow

**Key Insights:**
- Neural networks are just function composition
- Learning is gradient descent on weights
- OOP makes complex systems manageable
- Professional frameworks use the same principles

### What's Next: Tutorial F Part 2

In the next tutorial, we'll:
1. **Add more features**: Loss functions, optimizers, batch normalization
2. **Build a complete framework**: More like PyTorch
3. **Solve real problems**: Image classification, regression
4. **Compare architectures**: Shallow vs deep, different activations
5. **Understand design patterns**: How frameworks are organized

### Practice Exercises

Before moving on, try these:

1. **Modify activations**: Try using Sigmoid instead of ReLU. How does it change training?

2. **Change architecture**: Add another layer to the XOR network. Does it help?

3. **New problem**: Create a network for the OR gate (should be easier than XOR!)

4. **Visualize weights**: Plot the weights of a trained layer. What patterns do you see?

5. **Learning rate**: Experiment with different learning rates. What happens if it's too large or too small?

### Key Takeaway

You now understand **exactly** what happens inside `nn.Linear()` and why frameworks are designed the way they are. This foundation will make you a much more effective deep learning practitioner!

**Continue to Tutorial F Part 2 to build a complete neural network framework!**